# MIS-VQA: dataset

Loads MIS-VQA from Hugging Face, shows its contents on test frames and converts it to the LLaVA training format.

```bash
pip install datasets matplotlib pandas pillow
```

The images are not included. Request CholecT50 from [CAMMA](https://github.com/CAMMA-public/cholect50) and the EndoVis 2018 Robotic Scene Segmentation training set from the [challenge website](https://endovissub2018-roboticscenesegmentation.grand-challenge.org/), then set the two paths below.

In [ ]:
import os
import textwrap

import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset
from PIL import Image

ROOTS = {"cholect50": "CholecT50/videos",   # folder with VID01/, VID02/, ...
         "endovis18": "endovis2018/train"}  # folder with seq_1/, seq_2/, ...

data = {name: load_dataset("M-Hamdy/MIS-VQA", name) for name in ROOTS}
data

## Statistics

In [ ]:
TYPES = ["concise", "detailed", "conv1", "conv2", "reasoning"]
rows = []
for name, d in data.items():
    for split, s in d.items():
        rows.append({"dataset": name, "split": split, "frames": len(s),
                     **{t: len(s) if t in ("concise", "detailed") else sum(map(len, s[t])) for t in TYPES}})
table = pd.DataFrame(rows).set_index(["dataset", "split"])
table["total"] = table[TYPES].sum(axis=1)
table

## Examples

In [ ]:
def show(name, row, width=110):
    path = f"{ROOTS[name]}/{row['image']}"
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found; set ROOTS['{name}'] to the folder with the {name} images")
    image = Image.open(path)
    plt.figure(figsize=(6, 3.6))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"{name}  {row['image']}", fontsize=9)
    plt.show()
    print(textwrap.fill("Concise: " + row["concise"], width), "\n")
    print(textwrap.fill("Detailed: " + row["detailed"], width), "\n")
    for t in ("conv1", "conv2", "reasoning"):
        qa = row[t][0]
        print(textwrap.fill(f"{t}  Q: {qa['question']}", width))
        print(textwrap.fill(f"{' ' * len(t)}  A: {qa['answer']}", width), "\n")

show("cholect50", data["cholect50"]["test"][200])

In [ ]:
show("endovis18", data["endovis18"]["test"][100])

## LLaVA training format

Each description and each question-answer pair becomes one conversation. This is the input of the LLaVA-SurgXpert training commands in the [GitHub repository](https://github.com/M-Hamdy-M/MIS-VQA), with `--image_folder` set to the image folder above.

In [ ]:
PROMPTS = {"concise": "Describe the surgical scene concisely.", "detailed": "Describe the surgical scene in detail."}

def to_llava(split, types=TYPES):
    out = []
    for row in split:
        for t in types:
            pairs = [(PROMPTS[t], row[t])] if t in PROMPTS else [(qa["question"], qa["answer"]) for qa in row[t]]
            for q, a in pairs:
                out.append({"id": f"{t}_{len(out)}", "image": row["image"], "conversations": [
                    {"from": "human", "value": "<image>\n" + q}, {"from": "gpt", "value": a}]})
    return out

train = to_llava(data["endovis18"]["train"])
print(f"{len(train):,} conversations")
train[0]